# Session 9 — Evaluating Agent Trajectories

**Halvard Works maintenance assistant.** Planner → Diagnostics → Documentation → Maintenance.
Same plant, same four agents as Sessions 7 and 8. Nothing is rebuilt and no agent is re-run.

A **trajectory** is everything the system did on the way to an answer, rather than the answer
itself. The **path** is the shortest part of it — the ordered list of agents that ran. The glossary
below breaks a trajectory into its four parts; this session never looks at the answer at all.
Every trajectory here was captured on 13 September and is sitting in `runs7.json`.

> **The sentence this session adds:** *Code can tell you the path was wrong — but only if
> someone already wrote down what right was.*

Halvard Works is fictional — the plant, its machines, sensors, manuals, parts and history were
invented for this course. Inspired by publicly described industrial-copilot products; not
affiliated with or endorsed by any vendor.

## Every term, before it is used

| term | in this session it means |
|---|---|
| **trajectory** | everything the system DID on the way to an answer: the question, the plan, which agents ran in what order, and what was handed between them. Not the answer itself. |
| **path** | short for the ordered list of agents that ran. `planner → diagnostics → documentation`. |
| **plan** | the planner's decomposition: a list of (agent, subtask) pairs it decided on before anything ran. |
| **subtask** | one line of that plan — *what* a given agent was asked to do. |
| **agent_calls** | which agents actually ran, in order, including repeats. The path, as a list. |
| **handoff** | the single string one agent passes to the next. Nothing else crosses the boundary. |
| **payload** | the text inside a handoff. |
| **delegation row** | the hand-written record of what a given request's path SHOULD have been: `expected_agents`, `expected_calls`, `handoff_facts`. Session 7 wrote twelve of them. |
| **reference** | any answer key an evaluator is given. The delegation row is our reference. |
| **reference-free** | an evaluator that gets the trajectory and no answer key. The three judges in this session. |
| **arm** | one seeded version of a trajectory: `healthy`, `wrong_delegation`, `lost_handoff`, `redundant_call`, `delegation_loop`. |
| **seed** | the deterministic injector that produces an arm. The agents are byte-identical across all five. |
| **gate** | the test a judge has to pass before we trust it: fire on the arm it should stop, stay quiet elsewhere, by more than it disagrees with itself. |
| **separation** | how much more often a judge fires on the arm it should stop than on the arms it should pass. |
| **wobble** | how often the same judge, on the same unchanged trajectory, disagrees with its own most common answer. |
| **an evaluator fires** | it returned 0. Confusingly, that is the evaluator working. |
| **code evaluator** | a deterministic check written in Python. Session 7 wrote five. All five take the delegation row. |
| **expected_agents** | the row's field naming which agents should run, in order. |
| **expected_calls** | the row's field naming how many times each agent should run. `HW-006` says `diagnostics: 2`. |
| **handoff_facts** | the row's field naming text that MUST appear in a given handoff. |
| **healthy** | the unmodified arm. Every judge must pass it, or separation is measured against nothing. |
| **wrong_delegation** | the arm that hands a subtask to the wrong specialist. |
| **lost_handoff** | the arm that strips the `MACHINE:`/`FAULT_CODE:` tail off one payload. The path and the final answer are unchanged. |
| **redundant_call** | the arm that runs an agent twice for the *same* subtask. |
| **delegation_loop** | the arm that revisits agents with nothing new each time. |
| **redundant invocation** | an agent running when it had nothing new to contribute. Counted against the plan, never against "more than one". |
| **delegation_fit** | the judge that asks whether each subtask went to an agent whose job covers it. |
| **handoff_sufficiency** | the judge that asks whether each agent received what it needed. |
| **path_efficiency** | the judge that asks whether any invocation added nothing. |
| **absence** | a fact that is missing rather than wrong. Judges are measurably weak at spotting these. |
| **rule of three** | with zero events in n trials, the 95% upper bound on the rate is 3/n — not zero. |
| **USABLE** | a judge whose separation lower bound clears its wobble upper bound. |
| **DECORATION** | a judge whose separation does not clear its wobble. It may still be right; we cannot show it. |
| **NOT MEASURABLE** | there is no data that could answer the question — reported as itself, never as an interval. |
| **path-length bias** | firing more on long paths you should pass. This session's version of Session 8's verbosity bias. |
| **paired interval** | the confidence interval on a per-question difference, where both arms answered the same questions. |
| **margin of indifference** | a difference you declare too small to care about, BEFORE measuring. |
| **execution gap** | the plan says one thing and the run did another. Not a wrong plan and not a wrong answer — a run that did not honour its own decomposition. |
| **equivalence** | showing the whole interval sits inside that margin. Much harder than failing to find a difference. |

In [ ]:
# [PULL]
!git pull --ff-only
!python check_env.py

In [ ]:
# [SETUP]
import _path  # noqa: F401   puts shared/ and plant/ on sys.path — always first

import traj9, judge9, cost9, agree9, traj_bench9, coord_eval7
from delegation_rows7 import BY_ID

print('traj9  ', traj9.__version__)
print('judge9 ', judge9.__version__)
print('arms   ', traj9.ARMS)
print('gate   ', traj9.GATE_ROWS)

## 1. What a trajectory is

Session 1 said output-only evaluation is insufficient. Session 2 said the trace holds what the
answer cannot. Session 8 built judges that read the **report**. This session reads the **path**.

`traj9.render()` is the one place that decides what *seeing a trajectory* means here. Everything —
the judge prompts, the hands-on, this notebook — calls it, so the room and the model are always
looking at exactly the same thing.

Notice what is **not** in it: no final report, and no delegation row. Both omissions are deliberate
and both are explained below.

In [ ]:
# [LOOK]
t = traj9.trajectory('HW-001', 'healthy')
print(traj9.render(t))

## 2. Two paths. Both call `diagnostics` twice.

One of these is correct and one is waste. Look at the paths alone — `agent_calls` — and decide
which is which before running the next cell.

In [ ]:
# [HOOK]
a = traj9.trajectory('HW-006', 'healthy')          # a real request, unmodified
b = traj9.trajectory('HW-001', 'redundant_call')   # a seeded failure

for name, t in (('A', a), ('B', b)):
    print(f"{name}: {' -> '.join(t['agent_calls'])}")

print()
print('same shape?', [x for x in a['agent_calls'] if x != 'planner'] ==
                     [x for x in b['agent_calls'] if x != 'planner'])

Same shape. The path cannot tell you which is which — you have to read what each invocation
was *for*.

In [ ]:
# [HOOK-ANSWER]
for name, t in (('A', a), ('B', b)):
    print(name)
    for i, step in enumerate(t['plan'], 1):
        print(f"   {i}. {step['agent']:14s} <- {step['subtask']}")
    print()

**A** was *given* two different machines to diagnose. **B** was given the same one twice.

The difference is invisible in the path and legible in the plan. On the plan, A is fine.

Now look at what A actually did.

In [ ]:
# [EXHIBIT]
# HW-006's plan is right. Its execution is not — and we did not find this. The judges did,
# on 20 Sep, on a trajectory our own answer key labels healthy.
import re
parts = re.split(r'^\[(\w+)\]\s*$', a['answer'], flags=re.M)
secs = list(zip(parts[1::2], parts[2::2]))
diag = [b.strip() for n, b in secs if n == 'diagnostics']

print('request:', a['question'])
print('plan   :', [s['subtask'] for s in a['plan']])
print()
print('diagnostics sections produced :', len(diag))
print('...and they are identical     :', len(diag) == 2 and diag[0] == diag[1])
print('BLOWER appears in the answer  :', 'BLOWER' in a['answer'])
print('BLOWER appears in any handoff :',
      any('BLOWER' in h['payload'] for h in a['handoffs'][1:]))
print()
print('what Session 7\'s five code checks say about it:')
for k, v in coord_eval7.run_all(a, BY_ID['HW-006']).items():
    print(f'   {k:22s} {v["score"]}')

The request names **two** machines. The plan says diagnose both. The run produced the same
CONVEYOR diagnosis twice, and the word BLOWER never appears again — not in the answer, not
in any handoff after the first.

**And all five code checks pass it.** `agent_no_redundancy` passes because the delegation row
says `diagnostics: 2` and it ran twice. It never asks what each call was *for*.

That is an **execution gap**, and nothing built so far in this course can see it. Hold that
thought — it comes back at the end.

## 3. How the code knew

Session 7's `agent_no_redundancy` gets this right every time. Here is why.

In [ ]:
# [ROW]
row = BY_ID['HW-006']
for f in ('expected_agents', 'expected_calls', 'forbidden_agents', 'handoff_facts'):
    print(f'{f:18s} {row[f]}')

`expected_calls` says `diagnostics: 2`. Somebody typed that. Twelve times, by hand, one row per
request — and every one of Session 7's five evaluators reads it.

In [ ]:
# [CODE]
for name, t, rid in (('A (HW-006 healthy)', a, 'HW-006'),
                     ('B (HW-001 redundant)', b, 'HW-001')):
    res = coord_eval7.agent_no_redundancy(t, BY_ID[rid])
    print(f"{name:24s} score={res['score']}  {res['comment']}")

### The problem

Twelve rows for twelve questions. A production assistant answers twelve **thousand** questions,
and nobody has written a row for any of them.

So the question this session asks is not *code or judge*. It is: **how much of what the code does
survives taking the answer key away?**

```
code    trajectory + the row   ->  verdict      (Session 7)
judge   trajectory             ->  verdict      (Session 9)
```

## 4. Three judges, one per code check

| code (has the key) | judge (no key) | the arm it must stop |
|---|---|---|
| `delegation_accuracy` | `delegation_fit` | `wrong_delegation` |
| `handoff_integrity` | `handoff_sufficiency` | `lost_handoff` |
| `agent_no_redundancy` + `no_delegation_loop` | `path_efficiency` | `redundant_call`, `delegation_loop` |

Each judge receives the question, the agent roster and the trajectory. It never receives the row.
It also never receives the **final report** — a trajectory judge shown the finished answer is
Session 8's judge with a new label, and it would score well for reasons that have nothing to do
with reading a path.

Read the prompt. You cannot argue with a verdict whose prompt you have not seen, and arguing with
the verdict is the homework.

In [ ]:
# [JUDGE-PROMPT]
print(judge9.build_prompt('path_efficiency', b))

In [ ]:
# [JUDGE-STUB]
# The STUB judge: keyword matching, deterministic, free, and NOT a finding about judges.
# It exists so this cell runs with no key on any provider. Fooling it is not fooling a model.
for name, t in (('A (correct)', a), ('B (redundant)', b)):
    out = judge9.run_all(t, stub=True)
    print(name)
    for k, v in out.items():
        print(f"   {k:22s} {str(v['score']):>5s}  {v['comment']}")
    print()

## 5. Five seeded paths, already built and already run

**12 engineer requests × 5 arms × 3 repetitions = 180 captured trajectories.** Every request
is run through all five arms. The cell below shows one of the 12 — HW-001 — so you can see
what each arm does to a single path.

Session 7 built these as **state injectors**, not prompt edits — the agents are byte-identical
across all five arms, so anything an evaluator catches is coordination and nothing else.

They are also free: the `matrix` phase ran on a deterministic stub, which is why it carries
`latency_s` and `n_tool_calls` but **no token or cost figures at all**. Nothing in this session
prices a seeded failure.

In [ ]:
# [ARMS]
for arm in traj9.ARMS:
    t = traj9.trajectory('HW-001', arm)
    print(f"{arm:18s} {' -> '.join(t['agent_calls'])}")

In [ ]:
# [NO-MONEY]
# The guard that stops a seeded failure ever reaching a money slide.
try:
    traj9.assert_no_cost(traj9.trajectory('HW-001', 'healthy'))
except ValueError as e:
    print('refused, correctly:'); print(' ', e)

## 6. The arm you cannot see

Four of the five arms change the path. `lost_handoff` does not. It strips the
`MACHINE:` / `FAULT_CODE:` tail off one payload and changes nothing else.

Run this cell before reading on.

In [ ]:
# [INVISIBLE]
# What does each broken arm actually CHANGE? Measured on every row, every field.
rows = [r['row_id'] for r in traj9.runs('matrix', seed='healthy', rep=1)]
FIELDS = ('plan', 'agent_calls', 'handoffs', 'answer')

print('compared with the SAME request, unmodified — how many of the 12 requests',
      'the seed changes each field in:')
print()
print(f"{'arm':18s}" + ''.join(f'{f:>14s}' for f in FIELDS))
for arm in traj9.ARMS[1:]:
    n = {f: 0 for f in FIELDS}
    for r in rows:
        h, l = traj9.trajectory(r, 'healthy'), traj9.trajectory(r, arm)
        for f in FIELDS:
            n[f] += (h[f] != l[f])
    print(f'{arm:18s}' + ''.join(f'{n[f]:>10d}/12' for f in FIELDS))

Read the `lost_handoff` row against the other three. The other three change **everything** —
the plan, which agents ran, the handoffs and the final answer — on 9 or 10 of the 12 rows.
`lost_handoff` changes **one field, on 7 rows, and nothing else anywhere**.

The seed deletes exactly 43 characters from the end of one payload: the
`MACHINE:` / `FAULT_CODE:` tail. Everything before it is byte-identical. So `documentation`
was asked to find the procedure for a fault it was never told.

The final answer is **byte-identical** on every row. So no report-level judge — including all four
of Session 8's — can ever see this fault. Only something reading the handoffs can.

And a reference-free judge has to notice an **absence**: a fact that is not there, with no list of
facts that should have been. Session 8 measured exactly that weakness — the `hide_the_risk` attack
fooled a *certified* judge on the first try by **deleting** the sentence that stated the alarm limit.

So the prediction, written into `judge9.py` before the live run: **`handoff_sufficiency` will be this
session's DECORATION judge.** The next cells say whether that held.

In [ ]:
# [SEE-IT]
h = traj9.trajectory('HW-001', 'healthy')
l = traj9.trajectory('HW-001', 'lost_handoff')
for tag, t in (('healthy', h), ('lost_handoff', l)):
    edge = [x for x in t['handoffs'] if x['from'] == 'diagnostics'][0]
    print(f'--- {tag}: diagnostics -> {edge["to"]}')
    print(edge['payload'])
    print()

## 7. The bar, unchanged from Session 8

A judge is **USABLE** when the lower bound of its separation clears the upper bound of its wobble.
Otherwise it is **DECORATION** — it may still be right, we just cannot show it.

Zero flips does not mean zero wobble. It means the **rule of three**: with n=10 the bound is still
30%, and only pooling three trajectories takes it to 10%.

In [ ]:
# [GATE]
import json, pathlib
p = pathlib.Path('traj_runs9.json')
if p.exists():
    recs = traj_bench9.load(p.name)
else:
    print('no live verdicts yet — showing the STUB, which proves plumbing and nothing else\n')
    recs = traj_bench9.gate(stub=True, verbose=False)
    recs += traj_bench9.wobble(stub=True, n=3, verbose=False)

rep = agree9.report(recs)
# PER ARM. Pooling a judge's two target arms turned 4/4 on one and 0/4 on the other
# into 50%, which answers neither question. Changed after the run — see agree9.separation.
for j, arm in agree9.judge_arm_pairs():
    s, w = agree9.separation(recs, j, arm), agree9.wobble(recs, j)
    print(f"{j+' vs '+arm:42s} sep {s.diff:+4.0%} [{s.lo:+.0%}, {s.hi:+.0%}]   "
          f"wobble <= {w.upper:.0%}   -> {rep['verdicts'][j+'::'+arm].split(' —')[0]}")

print()
for k, why in rep['excluded'].items():
    print('excluded from the gate, scored in neither direction:')
    print(' ', k, '—', why)

In [ ]:
# [GRID]
# The raw calls behind the fractions. One judge, every arm, every request.
# ONE model call per cell — these are four different requests, not four repeats.
J = 'path_efficiency'
g = [r for r in recs if r['phase'] == 'gate' and r['judge'] == J]
print(f'{J}\n')
print(f"{'arm':20s}" + ''.join(f'{traj9.label(r)[:22]:>24s}' for r in traj9.GATE_ROWS))
for arm in traj9.ARMS:
    cells = []
    for row in traj9.GATE_ROWS:
        v = [x for x in g if x['arm'] == arm and x['row_id'] == row][0]
        cells.append('(excluded)' if traj9.is_excluded(row, arm)
                     else 'FIRED' if v['score'] == 0 else 'passed')
    tag = '  <- target' if arm in judge9.TARGETS[J] else ''
    print(f'{arm:20s}' + ''.join(f'{c:>24s}' for c in cells) + tag)

Every cell is **one model call**. `4/4` on the bottom row is four *different requests*,
one call each — not four repeats of the same one. Repeats are a separate 30 calls, and
they are the wobble number.

In [ ]:
# [BIAS]
# Session 8's bias was VERBOSITY: same report, four times the words, verdict moved.
# The trajectory analogue is PATH LENGTH — healthy runs 3 agents, delegation_loop runs 8.
for j, v in rep['path_length_bias'].items():
    if not v['measurable']:
        print(f"{j:22s} NOT MEASURABLE — every long path is one it should fire on")
    else:
        print(f"{j:22s} short {v['short_fires']}/{v['n_short']}  "
              f"long {v['long_fires']}/{v['n_long']}  "
              f"{v['diff']:+.0%} [{v['lo']:+.0%}, {v['hi']:+.0%}]")

### The one thing a judge found that five code checks did not

Back to HW-006 — the run from the start of this notebook, the one every code evaluator
passed. Nobody pointed a judge at it. It is labelled `healthy` in our own answer key.

In [ ]:
# [FOUND]
for v in rep['excluded_verdicts']:
    if v['verdict'] == 'UNSOUND':
        print(f"{v['judge']}:")
        print(' ', v['comment'])
        print()

**It is scored in neither direction — not a catch, not a false alarm.** Counting it as a
catch would define the test by what the judge found and then mark the judge on it. The
reason it is here at all is that you can verify the flaw from the record yourself, without
the judge — which is the bar for putting it on a slide.

So: does this mean judges beat code? No. **Code caught three failures the judges missed
entirely. The judges caught one thing code was never told to look for.** Different
instruments, different blind spots.

## 8. What a path costs

This half of the session uses the **live** `comparison` phase only: 12 rows × 3 reps × 2 arms,
claude-sonnet-5, 13 September. Those records carry all eight metrics.

Paired, because both arms answer the same twelve questions: subtract per question and the
row-to-row spread cancels. Sum into two totals and it does not — and the totals then hide the
only interesting thing in the data.

In [ ]:
# [COST]
runs = cost9.comparison_runs()
for m, v in cost9.table(runs).items():
    unit = '$' if m == 'cost_usd' else ''
    print(f"{m:16s} {unit}{v['diff']:+9.5f}  "
          f"[{unit}{v['lo']:+.5f}, {unit}{v['hi']:+.5f}]  {v['direction']}")

Latency separates comfortably. Cost separates by a hair — the lower bound is `$0.000053`.
Quality is a tie. **You can detect the time an org chart costs long before you can detect the money.**

In [ ]:
# [SPREAD]
for r in cost9.per_row_cost(runs):
    print(f"{r['pct']:+8.1f}%  {r['row_id']}  {r['path']}")

Three rows are **cheaper** with four agents, because the planner routes a narrow question to one
specialist while the single agent does everything. Two rows cost 2.7×. The mean hides all of it.

In [ ]:
# [EQUIV]
e = cost9.equivalence_n(runs)
print(f"margin +/-{e['margin']:.0%}, rows available {e['rows_available']}")
for r in e['ladder']:
    mark = '  EQUIVALENT' if r['equivalent'] else ''
    print(f"  n={r['n_per_arm']:5d}/arm  [{r['lo']:+.1%}, {r['hi']:+.1%}]{mark}")
print(f"\nfirst n that clears: {e['first_n_that_clears']} rows per arm")

*"We could not detect a difference"* and *"there is no difference"* are different sentences.
Saying the second needs a pre-declared **margin of indifference** and the whole interval inside it —
about **400 rows per arm** at these rates. There are twelve. So the session does not say it.

In [ ]:
# [SIGN]
# Session 7 printed outcome_match as -4.3%. Both readings, so nobody captions one as the other.
s = cost9.sign_trap(runs)
pp, rel = s['percentage_points'], s['relative_as_session7_printed_it']
print(f"{pp['diff']:+.1f} percentage points [{pp['lo']:+.1f}, {pp['hi']:+.1f}] — {pp['reads']}")
print(f"{rel['pct']:+.4f}% relative — {rel['reads']}")
print('reproduces the Session 7 deck exactly:', cost9.reproduces_session7(runs))

## 9. Hands-on — be the judge with no key, then write the key

Open **`my_path9.py`**. It is the only file you edit, it makes **zero model calls**, and it works
the same on Anthropic, OpenAI or Gemini because it never talks to any of them.

```
python my_path9.py --show        read the six cases
python my_path9.py --keys        the two live runs you write a key for
python screen_my_path.py         mark yourself
```

**Part 1.** Six trajectories, no delegation row. For each: `SOUND`, or `UNSOUND` plus which of
`delegation` / `handoff` / `efficiency` is wrong. One case is healthy. They are *not* evenly spread.

**Part 2.** Two live trajectories. Write what the path should have been. The screener runs Session 7's
`delegation_accuracy` with **your** key against the real run, then shows you the shipped row.

You are not marked on matching the shipped row. You are being shown how much of the verdict was
decided by whoever wrote it.

In [ ]:
# [HANDSON]
!python my_path9.py --show A

---

### What Session 10 inherits

Three reference-free judges with a measured gate, five seeded arms that cost nothing to re-run, and
one arm that is invisible in both the path and the answer.

> *Code can tell you the path was wrong — but only if someone already wrote down what right was.*